# Minkowski Distance Family

Companion notebook for: [Minkowski Distance Family](https://ml-viz.vercel.app/wiki/minkowski-distances)

We:
1. Compute all metrics for the worked example (1,2,3)→(4,6,3)
2. Visualise unit balls for p=1, 2, ∞ in 2-D
3. Show how KNN decision boundaries shift with p

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import make_blobs

plt.style.use('dark_background')
plt.rcParams.update({
    'axes.facecolor': '#1a1d27',
    'figure.facecolor': '#0f1117',
    'axes.edgecolor': '#3a3d4a',
    'grid.color': '#2a2d3a',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'font.size': 11,
})

## 1 · Worked example: all metrics for (1,2,3)→(4,6,3)

In [ ]:
x = np.array([1, 2, 3], dtype=float)
y = np.array([4, 6, 3], dtype=float)
gaps = np.abs(x - y)  # [3, 4, 0]

print(f"Coordinate gaps: {gaps}")
print()

for p in [1, 2, 3, 5, 10, 50, 100]:
    dp = (gaps**p).sum() ** (1/p)
    print(f"d_{p:3d} = {dp:.6f}")

print(f"d_inf = {gaps.max():.6f}  (Chebyshev)")
print()
print("Order: d_1 >= d_2 >= d_3 >= ... >= d_inf")

## 2 · Unit balls for p = 1, 2, ∞ in 2-D

The unit ball in 2-D is the set $\{\mathbf{x} : d_p(\mathbf{x}, \mathbf{0}) \le 1\}$.

- p=1: diamond  
- p=2: circle  
- p→∞: square

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

p_values = [1, 2, 5, 100]   # p=100 ≈ Chebyshev
colors   = ['#f59e0b', '#6366f1', '#34d399', '#f43f5e']

theta = np.linspace(0, 2*np.pi, 2000)

for ax, p, color in zip(axes, p_values, colors):
    # Parametrize the unit ball boundary: |cos θ|^p + |sin θ|^p = 1
    # Scale each direction: r = 1 / (|cos θ|^p + |sin θ|^p)^(1/p)
    cos_t = np.cos(theta)
    sin_t = np.sin(theta)
    r = 1.0 / (np.abs(cos_t)**p + np.abs(sin_t)**p) ** (1/p)
    bx = r * cos_t
    by = r * sin_t

    ax.fill(bx, by, alpha=0.3, color=color)
    ax.plot(bx, by, color=color, lw=2)
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.axhline(0, color='#3a3d4a', lw=0.8)
    ax.axvline(0, color='#3a3d4a', lw=0.8)
    label = f'p={p}' if p < 100 else 'p→∞ (Chebyshev)'
    ax.set_title(label)
    ax.grid(True, alpha=0.2)

plt.suptitle('Unit balls for different Minkowski p', y=1.02)
plt.tight_layout()
plt.show()

## 3 · KNN decision boundaries shift with p

Train KNN (k=5) on a 2-class dataset; vary p from 1 to ∞ and see how boundaries change.

In [ ]:
rng = np.random.default_rng(7)
X_train, y_train = make_blobs(n_samples=80, centers=2, cluster_std=1.2,
                               random_state=7)

xx, yy = np.meshgrid(np.linspace(X_train[:,0].min()-1, X_train[:,0].max()+1, 200),
                     np.linspace(X_train[:,1].min()-1, X_train[:,1].max()+1, 200))
grid = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, p in zip(axes, [1, 2, 5, 100]):
    label = f'p={p}' if p < 100 else 'p→∞'
    knn = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=p)
    knn.fit(X_train, y_train)
    Z = knn.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.25, cmap='coolwarm')
    ax.scatter(X_train[:,0], X_train[:,1], c=y_train,
               cmap='coolwarm', s=30, edgecolors='white', lw=0.5)
    ax.set_title(f'KNN k=5, {label}')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('KNN decision boundaries — p parameter effect', y=1.02)
plt.tight_layout()
plt.show()

---

## ✏️ Your turn

### Exercise 1 — distance from scratch

Implement `minkowski(x, y, p)` without using scipy and verify it matches the worked example.

In [ ]:
def minkowski(x, y, p):
    """Minkowski distance between 1-D arrays x and y."""
    # TODO(you): implement using only numpy
    pass

x = np.array([1, 2, 3], dtype=float)
y = np.array([4, 6, 3], dtype=float)

# assert abs(minkowski(x, y, 1) - 7.0) < 1e-9, "Manhattan should be 7"
# assert abs(minkowski(x, y, 2) - 5.0) < 1e-9, "Euclidean should be 5"
# assert abs(minkowski(x, y, 3) - 91**(1/3)) < 1e-6, "p=3 should be 91^(1/3)"
print("Uncomment asserts after implementing minkowski().")

### Exercise 2 — unit ball area vs p

In 2-D, the area of the unit ball for $\ell^p$ is $\frac{4\,\Gamma(1/p+1)^2}{\Gamma(2/p+1)}$. Compute and plot the area for p=0.5, 1, 2, 5, 10, ∞.

In [ ]:
from scipy.special import gamma

# TODO(you): compute and plot the area for several p values
# area(p) = 4 * gamma(1/p + 1)**2 / gamma(2/p + 1)
# Note: at p=inf, the unit ball is a square with area 4
ps = np.array([0.5, 0.75, 1, 1.5, 2, 3, 5, 10, 50])
# areas = ...
print("Fill in the areas and plot them.")

<details>
<summary>Solutions</summary>

```python
# Exercise 1
def minkowski(x, y, p):
    return (np.abs(x - y)**p).sum() ** (1/p)

assert abs(minkowski(x, y, 1) - 7.0) < 1e-9
assert abs(minkowski(x, y, 2) - 5.0) < 1e-9
assert abs(minkowski(x, y, 3) - 91**(1/3)) < 1e-6
print("All assertions pass!")

# Exercise 2
areas = 4 * gamma(1/ps + 1)**2 / gamma(2/ps + 1)
plt.figure(figsize=(7,4))
plt.plot(ps, areas, 'o-', color='#6366f1')
plt.axhline(4, color='#94a3b8', ls='--', label='Square area (p→∞) = 4')
plt.axhline(np.pi, color='#f59e0b', ls='--', label='Circle area (p=2) = π')
plt.legend()
plt.xlabel('p'); plt.ylabel('Unit ball area')
plt.title('2-D unit ball area vs p')
plt.show()
```
</details>